# Data Exploration: Fed-Heart-Disease


In [1]:
import os
import sys
from pathlib import Path

root_dir = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.data import get_site_summary, load_all_sites, load_pooled_data
from flamby.datasets.fed_heart_disease.metric import metric

config = {
    "dataset": "fed_heart_disease",
    "num_sites": 4,
    "plot_path": str(root_dir / "plots" / "site_sizes.png"),
}

Summary table showing sample counts, positive label counts, and disease prevalence across the four hospital sites.

In [2]:
summary_df = get_site_summary()
print(summary_df.to_string(index=False))

 site        name  n_train  n_test  n_total  pos_train  pos_test  prev_train  prev_test  prev_total
    0   cleveland      199     104      303         91        48       0.457      0.462       0.459
    1   hungarian      172      89      261         65        33       0.378      0.371       0.375
    2 switzerland       30      16       46         30        15       1.000      0.938       0.978
    3          va       85      45      130         66        35       0.776      0.778       0.777


Feature shapes, dimensions, and verification of tensor extraction from pooled dataset.

In [3]:
x_pool_tr, y_pool_tr = load_pooled_data(train=True)
x_pool_te, y_pool_te = load_pooled_data(train=False)
print(f"Pooled train: X={tuple(x_pool_tr.shape)}, y={tuple(y_pool_tr.shape)}")
print(f"Pooled test:  X={tuple(x_pool_te.shape)}, y={tuple(y_pool_te.shape)}")

Pooled train: X=(486, 13), y=(486, 1)


Pooled test:  X=(254, 13), y=(254, 1)


Official metric: binary classification accuracy. Center 2 has 100% positive train samples, making ROC AUC undefined.

In [4]:
# Metric in FLamby: binary classification accuracy ((y_pred > 0.5) == y_true).mean()
y_test_true = y_pool_te.numpy().ravel()
dummy_preds = np.ones_like(y_test_true) * 0.51
print("Official metric: binary classification accuracy")
print(f"Baseline majority predictor accuracy: {metric(y_test_true, dummy_preds):.4f}")

Official metric: binary classification accuracy
Baseline majority predictor accuracy: 0.5157


Bar plot of sample sizes and positive cases per hospital site saved to plots/site_sizes.png.

In [5]:
os.makedirs(os.path.dirname(config["plot_path"]), exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
x_pos = np.arange(len(summary_df))
width = 0.35

ax.bar(x_pos - width/2, summary_df["n_train"], width, label="Train Total", color="#2b5c8f")
ax.bar(x_pos - width/2, summary_df["pos_train"], width, label="Train Positive (Disease)", color="#d95f02", alpha=0.85)
ax.bar(x_pos + width/2, summary_df["n_test"], width, label="Test Total", color="#7570b3")
ax.bar(x_pos + width/2, summary_df["pos_test"], width, label="Test Positive", color="#e7298a", alpha=0.85)

ax.set_ylabel("Number of Patients")
ax.set_title("Fed-Heart-Disease: Sample and Class Distribution by Hospital")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['name']}\n(site {row['site']})" for _, row in summary_df.iterrows()])
ax.legend(frameon=True, fontsize=8)
plt.tight_layout()
plt.savefig(config["plot_path"], dpi=150)
plt.close()
print(f"Saved plot to {config['plot_path']}")

Saved plot to C:\Users\Soumadipta Konar\Desktop\fl-hospital-consortium\plots\site_sizes.png
